# Train TinyML Autoencoder cho WiFi Anomaly Detection

Notebook này train một autoencoder nhỏ để chạy trên ESP32.

## Model Architecture:
- Input: 6 features
- Encoder: 6 → 16 → 8
- Decoder: 8 → 16 → 6
- Total params: < 500 (very small for ESP32)

## Approach:
1. Train chỉ trên normal data (label=0)
2. Anomaly = high reconstruction error (MSE)
3. Convert to TensorFlow Lite Micro
4. Export as .h file cho ESP32

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

print(f"TensorFlow version: {tf.__version__}")

## 1. Load Dataset

In [ ]:
# Paths
BASE_DIR = Path('..').absolute()
DATA_FILE = BASE_DIR / 'data' / 'wifi_features_simple.csv'
MODEL_DIR = BASE_DIR / 'models'
MODEL_DIR.mkdir(exist_ok=True)

print(f"Loading data from: {DATA_FILE}")

# Load data
df = pd.read_csv(DATA_FILE)
print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())

print(f"\nLabel distribution:")
print(df['label'].value_counts())

## 2. Prepare Data

In [ ]:
# Separate features and labels
feature_cols = [col for col in df.columns if col != 'label']
X = df[feature_cols].values
y = df['label'].values

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Feature names: {feature_cols}")

# Split normal and anomaly
X_normal = X[y == 0]
X_anomaly = X[y == 1]

print(f"\nNormal samples: {len(X_normal)}")
print(f"Anomaly samples: {len(X_anomaly)}")

In [ ]:
# Standardize features
scaler = StandardScaler()
X_normal_scaled = scaler.fit_transform(X_normal)
X_anomaly_scaled = scaler.transform(X_anomaly)
X_all_scaled = scaler.transform(X)

# Split normal data into train/val
X_train, X_val = train_test_split(X_normal_scaled, test_size=0.2, random_state=42)

print(f"Training on {len(X_train)} normal samples")
print(f"Validation on {len(X_val)} normal samples")
print(f"Test on {len(X_anomaly_scaled)} anomaly samples")

## 3. Build Autoencoder Model

In [ ]:
# Model parameters
INPUT_DIM = X_train.shape[1]  # 6 features
ENCODING_DIM = 8              # Bottleneck dimension
HIDDEN_DIM = 16               # Hidden layer dimension

print(f"Input dimension: {INPUT_DIM}")
print(f"Hidden dimension: {HIDDEN_DIM}")
print(f"Encoding dimension: {ENCODING_DIM}")

In [ ]:
# Build model
def build_autoencoder(input_dim, hidden_dim, encoding_dim):
    """
    Build a simple autoencoder for anomaly detection
    Architecture: input_dim -> hidden_dim -> encoding_dim -> hidden_dim -> input_dim
    """
    # Encoder
    encoder_input = layers.Input(shape=(input_dim,), name='input')
    x = layers.Dense(hidden_dim, activation='relu', name='encoder_hidden')(encoder_input)
    encoded = layers.Dense(encoding_dim, activation='relu', name='encoded')(x)
    
    # Decoder
    x = layers.Dense(hidden_dim, activation='relu', name='decoder_hidden')(encoded)
    decoded = layers.Dense(input_dim, activation='linear', name='output')(x)
    
    # Full autoencoder
    autoencoder = models.Model(encoder_input, decoded, name='autoencoder')
    
    return autoencoder

# Create model
autoencoder = build_autoencoder(INPUT_DIM, HIDDEN_DIM, ENCODING_DIM)

# Compile
autoencoder.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

# Summary
autoencoder.summary()

## 4. Train Model

In [ ]:
# Training parameters
EPOCHS = 50
BATCH_SIZE = 32

# Callbacks
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# Train
history = autoencoder.fit(
    X_train, X_train,  # Autoencoder: input = output
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, X_val),
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training History')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='Train MAE')
plt.plot(history.history['val_mae'], label='Val MAE')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.title('Mean Absolute Error')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(MODEL_DIR / 'training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Evaluate & Find Threshold

In [ ]:
# Compute reconstruction errors
def compute_reconstruction_error(model, X):
    predictions = model.predict(X, verbose=0)
    mse = np.mean(np.square(X - predictions), axis=1)
    return mse

# Errors for normal (validation) data
normal_errors = compute_reconstruction_error(autoencoder, X_val)

# Errors for anomaly data
anomaly_errors = compute_reconstruction_error(autoencoder, X_anomaly_scaled)

print(f"Normal reconstruction error: mean={normal_errors.mean():.4f}, std={normal_errors.std():.4f}")
print(f"Anomaly reconstruction error: mean={anomaly_errors.mean():.4f}, std={anomaly_errors.std():.4f}")

In [ ]:
# Plot error distributions
plt.figure(figsize=(10, 5))

plt.hist(normal_errors, bins=50, alpha=0.5, label='Normal', color='blue')
plt.hist(anomaly_errors, bins=50, alpha=0.5, label='Anomaly', color='red')
plt.xlabel('Reconstruction Error (MSE)')
plt.ylabel('Frequency')
plt.title('Reconstruction Error Distribution')
plt.legend()
plt.grid(True)

# Suggest threshold (e.g., mean + 2*std of normal errors)
threshold = normal_errors.mean() + 2 * normal_errors.std()
plt.axvline(threshold, color='green', linestyle='--', linewidth=2, 
            label=f'Threshold={threshold:.4f}')
plt.legend()

plt.savefig(MODEL_DIR / 'error_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nSuggested threshold: {threshold:.4f}")

In [ ]:
# ROC curve to find optimal threshold
# Combine all labels
y_true = np.concatenate([np.zeros(len(normal_errors)), np.ones(len(anomaly_errors))])
y_scores = np.concatenate([normal_errors, anomaly_errors])

fpr, tpr, thresholds = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

# Plot ROC
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.grid(True)
plt.savefig(MODEL_DIR / 'roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

# Find optimal threshold (Youden's index)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]
print(f"Optimal threshold (Youden): {optimal_threshold:.4f}")
print(f"TPR: {tpr[optimal_idx]:.3f}, FPR: {fpr[optimal_idx]:.3f}")

In [ ]:
# Use optimal threshold for classification
THRESHOLD = optimal_threshold

# Predict
y_pred = (y_scores > THRESHOLD).astype(int)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Normal', 'Anomaly']))

## 6. Save Model (Keras Format)

In [ ]:
# Save Keras model
keras_model_path = MODEL_DIR / 'wifi_autoencoder.keras'
autoencoder.save(keras_model_path)
print(f"Saved Keras model to {keras_model_path}")

# Save scaler parameters
scaler_params = {
    'mean': scaler.mean_.tolist(),
    'scale': scaler.scale_.tolist(),
    'var': scaler.var_.tolist()
}

import json
scaler_path = MODEL_DIR / 'scaler_params.json'
with open(scaler_path, 'w') as f:
    json.dump(scaler_params, f, indent=2)
print(f"Saved scaler parameters to {scaler_path}")

# Save threshold
threshold_params = {
    'threshold': float(THRESHOLD),
    'auc': float(roc_auc)
}
threshold_path = MODEL_DIR / 'threshold.json'
with open(threshold_path, 'w') as f:
    json.dump(threshold_params, f, indent=2)
print(f"Saved threshold to {threshold_path}")

## 7. Convert to TensorFlow Lite

In [ ]:
# Convert to TFLite (float32 - no quantization)
# Note: Do NOT use converter.optimizations = [tf.lite.Optimize.DEFAULT]
# without a representative_dataset, as it produces float16 weights
# that require DEQUANTIZE ops unsupported by TFLite Micro on ESP32.
converter = tf.lite.TFLiteConverter.from_keras_model(autoencoder)

# Convert (pure float32 for TFLite Micro compatibility)
tflite_model = converter.convert()

# Save TFLite model
tflite_path = MODEL_DIR / 'wifi_autoencoder.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f"Saved TFLite model to {tflite_path}")
print(f"Model size: {len(tflite_model) / 1024:.2f} KB")

## 8. Test TFLite Model

In [ ]:
# Load TFLite model
interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
interpreter.allocate_tensors()

# Get input and output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input details:")
print(input_details)
print("\nOutput details:")
print(output_details)

In [ ]:
# Test inference on a few samples
def tflite_predict(interpreter, X):
    predictions = []
    for x in X:
        # Set input
        interpreter.set_tensor(input_details[0]['index'], x.reshape(1, -1).astype(np.float32))
        # Run inference
        interpreter.invoke()
        # Get output
        output = interpreter.get_tensor(output_details[0]['index'])
        predictions.append(output[0])
    return np.array(predictions)

# Test on a few samples
test_samples = X_all_scaled[:10]
keras_pred = autoencoder.predict(test_samples, verbose=0)
tflite_pred = tflite_predict(interpreter, test_samples)

# Compare
print("Comparison (Keras vs TFLite):")
for i in range(5):
    keras_mse = np.mean(np.square(test_samples[i] - keras_pred[i]))
    tflite_mse = np.mean(np.square(test_samples[i] - tflite_pred[i]))
    print(f"Sample {i}: Keras MSE={keras_mse:.6f}, TFLite MSE={tflite_mse:.6f}, Diff={abs(keras_mse - tflite_mse):.6f}")

## 9. Convert to C Header File (.h) for ESP32

In [ ]:
# Convert TFLite model to C array
def convert_to_c_array(tflite_model, var_name='g_wifi_model'):
    """
    Convert TFLite model bytes to C header file
    """
    # Convert to hex array
    hex_array = [f'0x{b:02x}' for b in tflite_model]
    
    # Format as C array
    c_array = ', '.join(hex_array)
    
    # Create header content
    header = f"""
// Auto-generated WiFi Anomaly Detection Model
// Model size: {len(tflite_model)} bytes
// Date: {pd.Timestamp.now()}

#ifndef WIFI_MODEL_H
#define WIFI_MODEL_H

const unsigned int {var_name}_len = {len(tflite_model)};
const unsigned char {var_name}[] = {{
  {c_array}
}};

#endif  // WIFI_MODEL_H
"""
    return header

# Generate header
c_header = convert_to_c_array(tflite_model, var_name='g_wifi_autoencoder_model')

# Save to file
header_path = MODEL_DIR / 'wifi_model.h'
with open(header_path, 'w') as f:
    f.write(c_header)

print(f"Saved C header file to {header_path}")
print(f"Model size: {len(tflite_model)} bytes")

## 10. Generate ESP32 Configuration File

In [ ]:
# Create configuration header for ESP32
config_header = f"""
// WiFi Anomaly Detection Configuration
// Auto-generated from training notebook

#ifndef WIFI_CONFIG_H
#define WIFI_CONFIG_H

// Model parameters
#define N_FEATURES {INPUT_DIM}
#define N_HIDDEN {HIDDEN_DIM}
#define N_ENCODED {ENCODING_DIM}

// Anomaly detection threshold
#define ANOMALY_THRESHOLD {THRESHOLD:.6f}f

// Feature scaler (mean)
const float SCALER_MEAN[N_FEATURES] = {{
  {', '.join([f'{m:.6f}f' for m in scaler.mean_])}
}};

// Feature scaler (scale)
const float SCALER_SCALE[N_FEATURES] = {{
  {', '.join([f'{s:.6f}f' for s in scaler.scale_])}
}};

// Feature names (for debugging)
const char* FEATURE_NAMES[N_FEATURES] = {{
  {', '.join([f'"{name}"' for name in feature_cols])}
}};

#endif  // WIFI_CONFIG_H
"""

# Save config header
config_path = MODEL_DIR / 'wifi_config.h'
with open(config_path, 'w') as f:
    f.write(config_header)

print(f"Saved configuration header to {config_path}")

## Summary

In [ ]:
print("\n" + "="*60)
print("MODEL TRAINING COMPLETE")
print("="*60)
print(f"Model architecture: {INPUT_DIM} → {HIDDEN_DIM} → {ENCODING_DIM} → {HIDDEN_DIM} → {INPUT_DIM}")
print(f"Total parameters: {autoencoder.count_params()}")
print(f"Model size: {len(tflite_model) / 1024:.2f} KB")
print(f"Anomaly threshold: {THRESHOLD:.6f}")
print(f"ROC AUC: {roc_auc:.4f}")
print(f"\nFiles generated:")
print(f"  1. {keras_model_path.name} - Keras model")
print(f"  2. {tflite_path.name} - TFLite model")
print(f"  3. {header_path.name} - C header for ESP32")
print(f"  4. {config_path.name} - Configuration header")
print(f"  5. {scaler_path.name} - Scaler parameters")
print(f"  6. {threshold_path.name} - Threshold parameters")
print("\nNext steps:")
print("  1. Copy wifi_model.h and wifi_config.h to ESP32 project")
print("  2. Include TensorFlow Lite Micro library in ESP32")
print("  3. Implement inference code in ESP32")
print("="*60)